# 006-segmentation-ceiling — acceptance checks

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from gt_query import boundary_damage, gt_labels, gt_query, stride_damage, substituted
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, label_probabilities, load_segmenter
from uavloc.geo import footprint_m
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H, class_ncc
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
LABEL_SOURCE = "osm"
MAP_DATE = "newest"
FRAMES_PER_PLACE_SHOWN = 3
DAMAGE_FRAME = "bushcha-00"
STRIDES = [1, 2, 4, 8, 16]
RADII_PX = [0, 1, 2, 4, 8]
SUBSTITUTED_CLASSES = ["building", "track"]
MIN_PRESENT_PX = 20               # a class enters the NCC check only if the GT query holds this many pixels of it
OVERLAY_ALPHA = 0.45
FOV_W_DEG = 60.0
MATCH_GSD_M = 0.385
DEVICE = "mps"
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
FRAMES_DATASET = "village-crossdate-frames"
EXP = "006-segmentation-ceiling"

# Data

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
labels = {place: gt_labels(place, LABEL_SOURCE) for place in PLACES}
figures = experiment_dir(EXP) / "figures"
figures.mkdir(exist_ok=True)

In [ ]:
def footprint(frame: DroneFrame) -> tuple[float, float]:
    return footprint_m(frame.altitude_m, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)

In [ ]:
def colours(query: np.ndarray) -> np.ndarray:
    """`(C, H, W)` → `(H, W, 3)` argmax class colours; pixels with no class are background."""
    return SEG_PALETTE[query.argmax(0)]

# GT query overlay

In [ ]:
def overlay_row(row: np.ndarray, frame: DroneFrame) -> None:
    """Frame, GT query, and the two blended, on the match grid."""
    gt = colours(gt_query(labels[frame_place(frame)], frame, footprint(frame), MATCH_GSD_M))
    image = np.asarray(frame.image().resize((gt.shape[1], gt.shape[0])))
    for ax, picture, title in [(row[0], image, "frame"), (row[1], gt, f"GT query ({LABEL_SOURCE})"),
                               (row[2], (image * (1 - OVERLAY_ALPHA) + gt * OVERLAY_ALPHA).astype(np.uint8), "overlay")]:
        ax.imshow(picture); ax.set_title(f"{frame.name}: {title}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])

In [ ]:
shown = [f for place in PLACES for f in [f for f in frames if frame_place(f) == place][:FRAMES_PER_PLACE_SHOWN]]
fig_overlay, axes = plt.subplots(len(shown), 3, figsize=(12, 1.9 * len(shown)))
for row, frame in zip(axes, shown):
    overlay_row(row, frame)
fig_overlay.tight_layout()
fig_overlay.savefig(figures / f"gt-query-overlay-{LABEL_SOURCE}.png", dpi=90)

# GT query vs GT map at the true pose

In [ ]:
gt_maps = {place: label_probabilities(labels[place], MATCH_GSD_M) for place in PLACES}

In [ ]:
def ncc_at_truth(frame: DroneFrame) -> dict[str, float]:
    """Per-class NCC of the GT query against the GT map patch cut at the same pose and size (one alignment)."""
    query = torch.from_numpy(gt_query(labels[frame_place(frame)], frame, footprint(frame), MATCH_GSD_M))
    patch = gt_maps[frame_place(frame)].crop_rotated(frame.position, frame.yaw_deg, *footprint(frame), out_mpp=MATCH_GSD_M)
    patch = torch.from_numpy(patch.transpose(2, 0, 1).copy())
    return {CLASSES[c]: float(class_ncc(query[c], patch[c])[0, 0]) for c in range(len(CLASSES))
            if MIN_PRESENT_PX <= query[c].sum() < query[c].numel()}

In [ ]:
ncc_table = pd.DataFrame({f.name: ncc_at_truth(f) for f in frames}).T
ncc_table.describe().round(3)

# GT map vs model map grid

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)
model_maps = {place: SegmentationFeatures(model, model, load_date(place, MAP_DATE), "all", MATCH_GSD_M, MATCH_GSD_M, DEVICE).probabilities
              for place in PLACES}

In [ ]:
pd.DataFrame({place: {"gt_shape": gt_maps[place].pixels.shape, "model_shape": model_maps[place].pixels.shape,
                      "same_corners": (gt_maps[place].top_left, gt_maps[place].bottom_right) ==
                                      (model_maps[place].top_left, model_maps[place].bottom_right)}
              for place in PLACES}).T

# Damage

In [ ]:
damage_frame = next(f for f in frames if f.name == DAMAGE_FRAME)
damage_gt = gt_query(labels[frame_place(damage_frame)], damage_frame, footprint(damage_frame), MATCH_GSD_M)
{"stride 1 is identity": bool(np.array_equal(stride_damage(damage_gt, 1), damage_gt)),
 "radius 0 is identity": bool(np.array_equal(boundary_damage(damage_gt, 0), damage_gt))}

In [ ]:
damage_levels = [(f"stride {s}", stride_damage(damage_gt, s)) for s in STRIDES] + \
                [(f"boundary r={r} px", boundary_damage(damage_gt, r)) for r in RADII_PX]
fig_damage, axes = plt.subplots(2, len(STRIDES), figsize=(3.2 * len(STRIDES), 4.2))
for ax, (title, query) in zip(axes.flat, damage_levels):
    ax.imshow(colours(query)); ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
fig_damage.suptitle(f"{DAMAGE_FRAME}: every damage level of the GT query")
fig_damage.tight_layout()
fig_damage.savefig(figures / "damage-levels.png", dpi=90)

# Channel substitution

In [ ]:
model_query = SegmentationFeatures.from_satellite_probabilities(model, gt_maps[frame_place(damage_frame)], "all", MATCH_GSD_M,
                                                                MATCH_GSD_M, DEVICE).drone_probabilities(damage_frame.image(), *footprint(damage_frame))
class_ids = [CLASSES.index(c) for c in SUBSTITUTED_CLASSES]
mixed = substituted(model_query, damage_gt, class_ids)
{"substituted equal GT": bool(np.array_equal(mixed[class_ids], damage_gt[class_ids])),
 "others equal model": bool(np.array_equal(np.delete(mixed, class_ids, 0), np.delete(model_query, class_ids, 0)))}